In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ronikdedhia/next-word-prediction")

print("Path to dataset files:", path)

/Users/rishuagrawal13/Desktop/nwp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 228k/228k [00:00<00:00, 282kB/s]

Extracting files...
Path to dataset files: /Users/rishuagrawal13/.cache/kagglehub/datasets/ronikdedhia/next-word-prediction/versions/1


In [6]:
import os

txt_file = None
for file in os.listdir(path):
    if file.endswith(".txt"):
        txt_file = os.path.join(path, file)

with open(txt_file, "r", encoding="utf-8") as f:
    text = f.read()


In [7]:
text = text.lower().strip()
faqs = text.split("\n")   # each line = one sentence


In [8]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [9]:
tokenizer = Tokenizer()

In [18]:
tokenizer.fit_on_texts(faqs)

In [19]:
len(tokenizer.word_index)

18503

In [20]:
vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary size:", vocab_size)

Vocabulary size: 18504


In [21]:
sequences = []

for sentence in faqs:
    token_list = tokenizer.texts_to_sequences([sentence])[0]
    if len(token_list) < 2:
        continue
    for i in range(1, len(token_list)):
        sequences.append(token_list[:i+1])

print("Total sequences:", len(sequences))


Total sequences: 101619


In [22]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

max_len = max(len(seq) for seq in sequences)
sequences = pad_sequences(sequences, maxlen=max_len, padding="pre")

X = sequences[:, :-1]
y = sequences[:, -1]

print(X.shape, y.shape)


(101619, 19) (101619,)


In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=100, input_length=max_len-1),
    LSTM(150, return_sequences=False),
    Dropout(0.2),
    Dense(vocab_size, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()


/Users/rishuagrawal13/Desktop/nwp/.venv/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [25]:
history = model.fit(
    X,
    y,
    epochs=5,
    batch_size=128,
    validation_split=0.1
)


Epoch 1/5
715/715 ━━━━━━━━━━━━━━━━━━━━ 44s 61ms/step - accuracy: 0.1087 - loss: 5.6435 - val_accuracy: 0.1068 - val_loss: 6.5911
Epoch 2/5
715/715 ━━━━━━━━━━━━━━━━━━━━ 43s 60ms/step - accuracy: 0.1269 - loss: 5.4204 - val_accuracy: 0.1153 - val_loss: 6.5492
Epoch 3/5
715/715 ━━━━━━━━━━━━━━━━━━━━ 42s 58ms/step - accuracy: 0.1385 - loss: 5.2372 - val_accuracy: 0.1241 - val_loss: 6.5705
Epoch 4/5
715/715 ━━━━━━━━━━━━━━━━━━━━ 40s 56ms/step - accuracy: 0.1492 - loss: 5.0786 - val_accuracy: 0.1248 - val_loss: 6.6103
Epoch 5/5
715/715 ━━━━━━━━━━━━━━━━━━━━ 41s 57ms/step - accuracy: 0.1584 - loss: 4.9339 - val_accuracy: 0.1273 - val_loss: 6.6426


In [26]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

def predict_next_word(text, model, tokenizer, max_len):
    sequence = tokenizer.texts_to_sequences([text])[0]
    sequence = pad_sequences([sequence], maxlen=max_len-1, padding="pre")
    predicted = np.argmax(model.predict(sequence, verbose=0), axis=-1)
    return tokenizer.index_word.get(predicted[0], "")


In [27]:
seed_text = "machine learning"
print("Next word:", predict_next_word(seed_text, model, tokenizer, max_len))


Next word: and


In [28]:
predict_next_word("deep learning is", model, tokenizer, max_len)
predict_next_word("neural networks are", model, tokenizer, max_len)


'a'

In [ ]:
predict_next_word("deep ", model, tokenizer, max_len)



'and'